# InterviewSense — Phase 3: Supervised Communication Model

This notebook is the **next step after Phase 2**. Phase 2 extracted communication-related features from Common Voice but deliberately did not invent communication labels.

This phase does four things:
1. Loads the Phase 2 feature bank.
2. Creates a human-rating template for valid communication labels.
3. When a labelled file is supplied, trains an interpretable **ElasticNet Regression** model and a **Random Forest Regression** comparison model.
4. Uses **speaker-independent GroupKFold cross-validation**, reports metrics, and exports the learned feature weights/importances.

**Important:** The notebook does not create fake labels. The final model is only trained when genuine human-rated labels are available.

In [ ]:
# Section 1 — Imports and configuration

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

RANDOM_SEED = 42

DATA_DIR = Path("data")
REPORT_DIR = Path("reports")
MODEL_DIR = Path("models")

DATA_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Phase 3 configuration loaded.")

In [ ]:
# Section 2 — Locate the Phase 2 feature dataset

# Preferred: copy the Phase 2 CSV into ./data/
candidates = sorted(DATA_DIR.glob("communication_features_*.csv"))

# Fallback: allow the user to specify the exact file manually.
PHASE2_FEATURE_FILE = ""

if PHASE2_FEATURE_FILE:
    feature_path = Path(PHASE2_FEATURE_FILE)
elif candidates:
    feature_path = candidates[-1]
else:
    raise FileNotFoundError(
        "Phase 2 feature CSV was not found. Put communication_features_*.csv "
        "inside the 'data' folder or set PHASE2_FEATURE_FILE to its exact path."
    )

features = pd.read_csv(feature_path)

print("Loaded:", feature_path)
print("Shape:", features.shape)
print("Columns:", len(features.columns))
display(features.head(3))

In [ ]:
# Section 3 — Select features for the communication model

# We deliberately do NOT use raw identifiers, transcript text, age/gender,
# or any existing assessment score as predictors.
#
# These are the interpretable communication-related groups extracted in Phase 2.

preferred_features = [
    # Speech rate
    "wpm_speech_duration",
    "words_per_second",

    # Speech activity / pauses
    "speech_ratio",
    "silence_ratio",
    "pause_ratio",
    "pause_count",
    "mean_pause_duration",
    "median_pause_duration",
    "std_pause_duration",
    "maximum_pause_duration",
    "pauses_per_minute",
    "long_pause_count",
    "long_pause_ratio",

    # Fillers
    "total_filler_count",
    "fillers_per_100_words",
    "strict_filler_count",
    "strict_fillers_per_100_words",

    # Linguistic complexity
    "word_count",
    "unique_word_count",
    "average_word_length",
    "type_token_ratio",
    "sentence_count",
    "average_words_per_sentence",

    # Prosody / voice
    "pitch_mean",
    "pitch_std",
    "pitch_range",
    "voiced_frame_ratio",

    # Energy
    "rms_mean",
    "rms_std",
    "rms_range",

    # Spectral
    "spectral_centroid_mean",
    "spectral_centroid_std",
    "spectral_bandwidth_mean",
    "spectral_bandwidth_std",
]

# Add MFCC statistics if they exist.
mfcc_features = [
    c for c in features.columns
    if c.startswith("mfcc_") and (c.endswith("_mean") or c.endswith("_std"))
]
preferred_features.extend(mfcc_features)

feature_cols = [c for c in preferred_features if c in features.columns]

if len(feature_cols) < 5:
    raise RuntimeError(
        "Too few expected Phase 2 features were found. "
        "Check that the Phase 2 feature CSV is being loaded."
    )

print("Selected predictor count:", len(feature_cols))
print(feature_cols)

In [ ]:
# Section 4 — Create the HUMAN RATING TEMPLATE
#
# This is the most important step before model training.
# Common Voice does not contain our desired communication-quality target,
# so we create a file that a human rater must fill using the same rubric.

rating_template = features[[
    c for c in ["_record_id", "filename", "audio_path", "speaker_id", "transcript"]
    if c in features.columns
]].copy()

# Human-rated targets.
rating_template["fluency_score"] = np.nan
rating_template["clarity_score"] = np.nan
rating_template["delivery_score"] = np.nan
rating_template["overall_communication_score"] = np.nan

# Optional qualitative notes for auditability.
rating_template["rater_notes"] = ""

template_path = DATA_DIR / "communication_human_ratings_template.csv"
rating_template.to_csv(template_path, index=False)

print("Human-rating template created:")
print(template_path)
print()
print("Rating scale: 1–5")
print("1 = very poor")
print("2 = poor")
print("3 = average")
print("4 = good")
print("5 = excellent")
print()
print("Recommended rating definitions:")
print("- Fluency: smoothness, hesitation, excessive pauses, and filler usage.")
print("- Clarity: intelligibility and ease of understanding.")
print("- Delivery: pace, rhythm, prosody, and overall speaking manner.")
print("- Overall communication: holistic human judgement of the recording.")

## Section 5 — Human annotation requirement

**Stop here if the rating CSV has not been completed.**

The model must not be trained on labels generated from the same acoustic features. The labels should come from independent human raters (ideally at least 2 raters per recording, with disagreements retained for analysis).

After annotation, save the completed file as `data/communication_human_ratings.csv`.

In [ ]:
# Section 6 — Load genuine human labels
#
# This cell intentionally stops before training when labels are absent.

LABEL_FILE = DATA_DIR / "communication_human_ratings.csv"

if not LABEL_FILE.exists():
    print("No human-labelled file found yet.")
    print("Expected:", LABEL_FILE)
    print("Complete the generated rating template, save it with this name,")
    print("then rerun this cell and continue with the training sections.")
    labels_ready = False
else:
    labels = pd.read_csv(LABEL_FILE)
    labels_ready = True

    print("Human labels loaded:", LABEL_FILE)
    print("Shape:", labels.shape)
    display(labels.head(3))

In [ ]:
# Section 7 — Validate and merge human labels

if not labels_ready:
    print("Training is skipped because genuine labels are not available.")
else:
    required = [
        "_record_id",
        "overall_communication_score"
    ]

    missing = [c for c in required if c not in labels.columns]
    if missing:
        raise ValueError(f"Missing required label columns: {missing}")

    labels = labels.drop_duplicates("_record_id", keep="last").copy()
    labels["overall_communication_score"] = pd.to_numeric(
        labels["overall_communication_score"], errors="coerce"
    )

    model_df = features.merge(
        labels[["_record_id", "overall_communication_score"]],
        on="_record_id",
        how="inner"
    )

    model_df = model_df.dropna(
        subset=["overall_communication_score"]
    ).reset_index(drop=True)

    # Keep only valid 1–5 human ratings.
    model_df = model_df[
        model_df["overall_communication_score"].between(1, 5)
    ].copy()

    print("Labelled recordings:", len(model_df))
    print("Score range:",
          model_df["overall_communication_score"].min(),
          "to",
          model_df["overall_communication_score"].max())

    if "speaker_id" not in model_df.columns:
        raise RuntimeError(
            "speaker_id is required for speaker-independent validation."
        )

    model_df["speaker_id"] = model_df["speaker_id"].astype(str)
    print("Unique speakers:", model_df["speaker_id"].nunique())

    if model_df["speaker_id"].nunique() < 6:
        raise RuntimeError(
            "At least 6 speakers are recommended for 5-fold GroupKFold."
        )

In [ ]:
# Section 8 — Speaker-independent 80/20 holdout split

if labels_ready:
    from sklearn.model_selection import GroupShuffleSplit

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=RANDOM_SEED
    )

    train_idx, test_idx = next(
        gss.split(
            model_df,
            groups=model_df["speaker_id"]
        )
    )

    train_df = model_df.iloc[train_idx].reset_index(drop=True)
    test_df = model_df.iloc[test_idx].reset_index(drop=True)

    train_speakers = set(train_df["speaker_id"])
    test_speakers = set(test_df["speaker_id"])

    print("Train rows:", len(train_df))
    print("Test rows :", len(test_df))
    print("Train speakers:", len(train_speakers))
    print("Test speakers :", len(test_speakers))
    print("Speaker overlap:", len(train_speakers & test_speakers))

    assert not (train_speakers & test_speakers),                 "Speaker leakage detected!"

In [ ]:
# Section 9 — Primary model: ElasticNet Regression
#
# ElasticNet is selected as the primary model because it gives us
# interpretable learned weights while handling correlated speech features.

if labels_ready:
    X_train = train_df[feature_cols]
    y_train = train_df["overall_communication_score"].values
    groups_train = train_df["speaker_id"].values

    elastic_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            max_iter=10000,
            random_state=RANDOM_SEED
        ))
    ])

    param_grid = {
        "model__alpha": [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
    }

    gkf = GroupKFold(n_splits=5)

    search = GridSearchCV(
        elastic_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=gkf,
        n_jobs=-1
    )

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    elastic_best = search.best_estimator_

    print("Best ElasticNet parameters:")
    print(search.best_params_)
    print("Best CV RMSE:", -search.best_score_)

In [ ]:
# Section 10 — Cross-validation: ElasticNet vs Random Forest

if labels_ready:
    X = train_df[feature_cols]
    y = train_df["overall_communication_score"].values
    groups = train_df["speaker_id"].values

    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    models = {
        "ElasticNet": elastic_best,
        "RandomForest": rf
    }

    rows = []

    for model_name, base_model in models.items():
        fold_mae, fold_rmse, fold_r2 = [], [], []

        for fold, (tr, va) in enumerate(
            gkf.split(X, y, groups=groups), start=1
        ):
            model = base_model

            model.fit(X.iloc[tr], y[tr])
            pred = model.predict(X.iloc[va])

            fold_mae.append(
                mean_absolute_error(y[va], pred)
            )
            fold_rmse.append(
                np.sqrt(mean_squared_error(y[va], pred))
            )
            fold_r2.append(
                r2_score(y[va], pred)
            )

        rows.append({
            "model": model_name,
            "CV_MAE_mean": np.mean(fold_mae),
            "CV_MAE_std": np.std(fold_mae),
            "CV_RMSE_mean": np.mean(fold_rmse),
            "CV_RMSE_std": np.std(fold_rmse),
            "CV_R2_mean": np.mean(fold_r2),
            "CV_R2_std": np.std(fold_r2),
        })

    cv_results = pd.DataFrame(rows)
    display(cv_results)
    cv_results.to_csv(
        REPORT_DIR / "communication_model_cv_comparison.csv",
        index=False
    )

In [ ]:
# Section 11 — Final unseen-speaker test evaluation

if labels_ready:
    elastic_best.fit(train_df[feature_cols],
                     train_df["overall_communication_score"])
    rf.fit(train_df[feature_cols],
           train_df["overall_communication_score"])

    test_y = test_df["overall_communication_score"].values

    predictions = {
        "ElasticNet": elastic_best.predict(test_df[feature_cols]),
        "RandomForest": rf.predict(test_df[feature_cols])
    }

    test_rows = []

    for name, pred in predictions.items():
        test_rows.append({
            "model": name,
            "Test_MAE": mean_absolute_error(test_y, pred),
            "Test_RMSE": np.sqrt(mean_squared_error(test_y, pred)),
            "Test_R2": r2_score(test_y, pred)
        })

    test_results = pd.DataFrame(test_rows)
    display(test_results)
    test_results.to_csv(
        REPORT_DIR / "communication_model_test_comparison.csv",
        index=False
    )

In [ ]:
# Section 12 — Extract the actual learned weights
#
# These are the parameters you can show in the technical presentation.

if labels_ready:
    elastic_model = elastic_best.named_steps["model"]

    weights = pd.DataFrame({
        "feature": feature_cols,
        "weight": elastic_model.coef_,
        "absolute_weight": np.abs(elastic_model.coef_)
    }).sort_values(
        "absolute_weight",
        ascending=False
    ).reset_index(drop=True)

    print("ElasticNet learned feature weights:")
    display(weights.head(25))

    weights.to_csv(
        REPORT_DIR / "communication_elasticnet_feature_weights.csv",
        index=False
    )

    print("Intercept:", elastic_model.intercept_)

In [ ]:
# Section 13 — Random Forest feature importance

if labels_ready:
    rf_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf.feature_importances_
    }).sort_values(
        "importance",
        ascending=False
    ).reset_index(drop=True)

    display(rf_importance.head(25))

    rf_importance.to_csv(
        REPORT_DIR / "communication_randomforest_feature_importance.csv",
        index=False
    )

In [ ]:
# Section 14 — Save final models

if labels_ready:
    import joblib

    joblib.dump(
        elastic_best,
        MODEL_DIR / "interviewsense_elasticnet_communication_model.joblib"
    )

    joblib.dump(
        rf,
        MODEL_DIR / "interviewsense_randomforest_communication_model.joblib"
    )

    print("Saved:")
    print(MODEL_DIR / "interviewsense_elasticnet_communication_model.joblib")
    print(MODEL_DIR / "interviewsense_randomforest_communication_model.joblib")

## Section 15 — What this phase proves

- Phase 2 supplies the measurable speech features.
- Human annotation supplies the independent communication target.
- ElasticNet provides interpretable feature weights.
- Random Forest provides a nonlinear comparison model.
- GroupKFold evaluates both models on unseen speakers inside the training data.
- The final 20% speaker-held-out test set gives a final unbiased evaluation.

**Do not claim the model measures communication skill until the human-labelled dataset exists and the test results are acceptable.**